# P02 - Normalizacion global mel y ventana temporal 1024

Este notebook registra las dos mejoras nuevas de preprocesamiento que si movieron Kaggle en la ronda de `investigation/results/`.

- `globalmel`: normalizacion global por banda mel aprendida solo en curated train.
- `f1024`: ventana temporal mas larga para la rama separable temporal.

Juntas habilitaron el mejor blend actual: private LB `0.67025`.


## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import zipfile

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

ROOT = Path.cwd()
if ROOT.name == '02_preprocesamiento':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / '02_preprocesamiento' / 'results'
INVESTIGATION = ROOT / 'investigation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(INVESTIGATION))

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)

from scripts.fat2019.data import CORRUPT_OR_BAD_LABEL_FILES, split_labels
from scripts.fat2019.features import read_wav_mono, log_mel_spectrogram, extract_log_mel_stats

from numpy.lib.format import read_magic, _read_array_header


def npz_array_headers(path: Path) -> pd.DataFrame:
    rows = []
    with zipfile.ZipFile(path) as zf:
        for info in zf.infolist():
            if not info.filename.endswith('.npy'):
                continue
            with zf.open(info) as f:
                version = read_magic(f)
                shape, fortran_order, dtype = _read_array_header(f, version)
            rows.append({
                'array': info.filename.replace('.npy', ''),
                'shape': str(shape),
                'dtype': str(dtype),
                'compressed_mb': round(info.compress_size / 1024**2, 2),
            })
    return pd.DataFrame(rows)


## Inspeccion de caches promovidos


In [2]:
caches = {
    'curated_f512_globalmel': DATA_DIR / 'curated_logmel_image_m128_f512_globalmel.npz',
    'test_f512_globalmel': DATA_DIR / 'test_logmel_image_m128_f512_globalmel.npz',
    'curated_f1024': DATA_DIR / 'curated_logmel_image_m128_f1024.npz',
    'test_f1024': DATA_DIR / 'test_logmel_image_m128_f1024.npz',
}
summary = pd.concat([
    npz_array_headers(path).assign(cache=name, path=str(path.relative_to(ROOT)))
    for name, path in caches.items()
], ignore_index=True)
display(summary[['cache', 'array', 'shape', 'dtype', 'compressed_mb']])
summary.to_csv(RESULTS_DIR / 'P02_globalmel_f1024_summary.csv', index=False)


,cache,array,shape,dtype,compressed_mb
0,curated_f512_globalmel,x,"(4964, 128, 512)",float16,338.78
1,curated_f512_globalmel,fnames,"(4964,)",<U12,0.03
2,curated_f512_globalmel,normalization,(),<U10,0.00
3,curated_f512_globalmel,global_mean,"(128,)",float32,0.00
4,curated_f512_globalmel,global_std,"(128,)",float32,0.00
5,test_f512_globalmel,x,"(3361, 128, 512)",float16,280.87
6,test_f512_globalmel,fnames,"(3361,)",<U12,0.02
7,test_f512_globalmel,normalization,(),<U10,0.00
8,test_f512_globalmel,global_mean,"(128,)",float32,0.00
9,test_f512_globalmel,global_std,"(128,)",float32,0.00


## Evidencia de la ronda nueva


In [3]:
evidence = pd.DataFrame([
    {'candidate': 'current835_sep_temporal_full165', 'change': 'referencia previa temporal', 'private_lb': 0.65928, 'decision': 'baseline'},
    {'candidate': 'current645_globalmel_sep_temporal_full355', 'change': 'agrega globalmel', 'private_lb': 0.66561, 'decision': 'keep'},
    {'candidate': 'current550_globalmel250_f1024_200', 'change': 'globalmel + f1024 sin SE', 'private_lb': 0.66996, 'decision': 'keep'},
    {'candidate': 'current475_globalmel200_se125_f1024_200', 'change': 'globalmel + SE + f1024', 'private_lb': 0.67025, 'decision': 'selected_final'},
    {'candidate': 'current835_sep_temporal_tta1024_full165', 'change': 'TTA multicrop', 'private_lb': 0.65835, 'decision': 'discard'},
])
evidence['delta_vs_reference'] = evidence['private_lb'] - 0.65928
display(evidence.sort_values('private_lb', ascending=False))
evidence.to_csv(RESULTS_DIR / 'P02_globalmel_f1024_evidence.csv', index=False)


,candidate,change,private_lb,decision,delta_vs_reference
3,current475_globalmel200_se125_f1024_200,globalmel + SE + f1024,0.67025,selected_final,0.01097
2,current550_globalmel250_f1024_200,globalmel + f1024 sin SE,0.66996,keep,0.01068
1,current645_globalmel_sep_temporal_full355,agrega globalmel,0.66561,keep,0.00633
0,current835_sep_temporal_full165,referencia previa temporal,0.65928,baseline,0.00000
4,current835_sep_temporal_tta1024_full165,TTA multicrop,0.65835,discard,-0.00093


## Decision

- `globalmel` queda promovido: mejora clara y defendible por normalizacion.
- `f1024` queda promovido solo con entrenamiento especifico para esa ventana.
- TTA multi-crop no se promueve: empeoro contra la referencia.
- La variante final usa estas decisiones dentro del ensemble, no como reemplazo aislado.
